# Preprocessing

Goal: predict **price**.

In [1]:
import numpy as np
import pandas as pd
from load import FLAGS, NUMERIC, load
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

df = load()

## 1. Pick columns

- drop `url`, `name`, `colour`
- material: first one only
- rare brands and materials: `other`

In [2]:
def group_rare(values, min_count=5):
    counts = values.value_counts()
    return values.mask(values.isin(counts[counts < min_count].index), "other")


data = df.drop(columns=["url", "name", "colour"])
data["brand"] = group_rare(data["brand"].astype(str))
data["material"] = group_rare(df["material"].str[0])
data.head()

,price,brand,length_cm,weight_g,material,lumens,rechargeable,tactical,edc
0,139.95,Nitecore,10.80,200.0,aluminum,8000.0,True,True,True
1,59.95,Fenix,6.70,36.0,aluminum,1200.0,True,False,False
2,136.99,Ledlenser,15.10,206.0,aluminum,400.0,False,False,False
3,149.95,Nitecore,14.15,116.0,aluminum,4000.0,False,True,False
4,58.99,Maglite,43.40,596.0,aluminum,151.0,False,False,False


In [3]:
data.isna().sum()[lambda s: s > 0].to_frame("missing")

,missing
length_cm,6
weight_g,1
material,4
lumens,3


## 2. Variant A: categories

- gaps: brand median, most common material
- numbers: low / mid / high

In [4]:
a = data.copy()
for column in ["length_cm", "weight_g", "lumens"]:
    a[column] = a[column].fillna(a.groupby("brand")[column].transform("median"))
a["material"] = a["material"].fillna(a["material"].mode()[0])

for column in NUMERIC:
    a[column] = pd.qcut(a[column], q=3, labels=["low", "mid", "high"])
a[FLAGS] = a[FLAGS].map(lambda v: "yes" if v else "no")
a.head()

,price,brand,length_cm,weight_g,material,lumens,rechargeable,tactical,edc
0,high,Nitecore,low,high,aluminum,high,yes,yes,yes
1,low,Fenix,low,low,aluminum,mid,yes,no,no
2,high,Ledlenser,high,high,aluminum,low,no,no,no
3,high,Nitecore,mid,mid,aluminum,high,no,yes,no
4,low,Maglite,high,high,aluminum,low,no,no,no


In [5]:
# bin edges (low | mid | high)
data[NUMERIC].quantile([1 / 3, 2 / 3]).round(1)

,price,length_cm,weight_g,lumens
0.333333,61.2,13.0,89.0,625.0
0.666667,116.3,14.9,180.0,2000.0


## 3. Variant B: numbers

- log, one-hot, scale to 0–1
- gaps: KNN

In [ ]:
b = data.copy()
target = b.pop("price")
b[["length_cm", "weight_g", "lumens"]] = np.log1p(
    b[["length_cm", "weight_g", "lumens"]]
)
b["material"] = b["material"].fillna("unknown")
b = pd.get_dummies(b, columns=["brand", "material"], dtype=int)
b[FLAGS] = b[FLAGS].astype(int)

# Impute predictors only, keep price out of the KNN distance matrix.
scaled = MinMaxScaler().fit_transform(b)
b = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(scaled), columns=b.columns)
price_scaler = MinMaxScaler()
b.insert(
    0,
    "price",
    price_scaler.fit_transform(np.log1p(target).to_numpy().reshape(-1, 1)).ravel(),
)
b.head()

,price,length_cm,weight_g,lumens,rechargeable,tactical,edc,brand_Fenix,brand_Ledlenser,brand_Maglite,brand_Nitecore,brand_Streamlight,brand_Wuben,brand_other,material_aluminum,material_other,material_plastic,material_polycarbonate,material_unknown
0,0.592132,0.421124,0.498459,0.873851,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.350989,0.234646,0.093751,0.625946,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.586028,0.556860,0.505493,0.482549,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.611848,0.530291,0.369056,0.783257,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.346423,1.000000,0.758784,0.355737,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## 4. Save

In [7]:
print("missing:", a.isna().sum().sum(), b.isna().sum().sum())
print("B range:", b.min().min(), b.max().max())

a.to_csv("data_categorical.csv", index=False)
b.round(4).to_csv("data_numeric.csv", index=False)
print("A:", a.shape, "B:", b.shape)

missing: 0 0
B range: 0.0 1.0
A: (200, 9) B: (200, 19)
